# 05 — Report-Quality Figures

**Objective:** Generate publication-ready charts for the 10-page final report.

All figures saved to `reports/figures/` at 200+ DPI.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from src.conventions import *
from src.optimizer import allocate_erc, allocate_vol_target, allocate_equal_weight, allocate_marp_replication
from src.backtest import run_backtest
from src.metrics import (
    annualised_return, annualised_vol, max_drawdown, sortino_ratio,
    calmar_ratio, tracking_error, information_ratio, factor_regression
)
from src.viz import set_style

# Ensure figures directory exists
FIGURES.mkdir(parents=True, exist_ok=True)
set_style()

In [ ]:
# Load data and run backtest (cached like a research report build step)
returns = pd.read_parquet(DATA_CLEAN / 'returns.parquet')
marp_full = pd.read_parquet(DATA_CLEAN / 'marp_official.parquet')
factors = pd.read_parquet(DATA_FACTORS / 'china_ff3_proxy.parquet')

marp_date = pd.to_datetime(marp_full['日期'])
marp_s = pd.Series(marp_full['收盘'].astype(float).values, index=marp_date).sort_index()
marp_ret = marp_s.pct_change().dropna()

assets = ['510300', '510500', '511010', '518880', '159980']
asset_labels = ['CSI 300', 'CSI 500', '5Y Treasury', 'Gold', 'Commodity']
asset_returns = returns[assets].dropna()

strat_dict = {
    'ERC_10':   lambda r, m: allocate_erc(r, vol_target=0.10),
    'RP_10':    lambda r, m: allocate_vol_target(r, vol_target=0.10),
    'MARP_rep': lambda r, m: allocate_marp_replication(r, m, method='ridge') if m is not None else allocate_equal_weight(r),
    'Equal':    lambda r, m: allocate_equal_weight(r),
}
try:
    from src.optimizer import allocate_hrp
    strat_dict['HRP_10'] = lambda r, m: allocate_hrp(r, vol_target=0.10)
except ImportError:
    pass

results = run_backtest(asset_returns, marp_returns=marp_ret, lookback=756, rebal_days=252, strategies=strat_dict)
print('Build complete.')

# MARP OOS benchmark
marp_oos = marp_ret.loc[OOS_START:].dropna()
marp_cum = (1 + marp_oos).cumprod()

## Figure 1: Cumulative Performance (Main Result)

In [ ]:
from src.viz import plot_cumulative_returns

fig = plot_cumulative_returns(
    results,
    benchmark_series=marp_ret,
    benchmark_label='CSI MARP 930929',
    title='A Tradable Multi-Asset Risk Parity Strategy vs CSI MARP 930929',
    save_path=str(FIGURES / 'fig1_cumulative.png'),
    figsize=(13, 6.5),
)
plt.show()

## Figure 2: Drawdown Analysis

In [ ]:
from src.viz import plot_drawdowns

fig = plot_drawdowns(results, save_path=str(FIGURES / 'fig2_drawdown.png'))
plt.show()

## Figure 3: Rolling Sharpe & Volatility

In [ ]:
from src.viz import plot_rolling_metrics

fig = plot_rolling_metrics(results, window=252, save_dir=str(FIGURES))
plt.show()

## Figure 4: Weight Heatmap (ERC_10)

In [ ]:
from src.viz import plot_weight_heatmap

fig = plot_weight_heatmap(
    results['ERC_10'],
    save_path=str(FIGURES / 'fig4_weights.png'),
)
plt.show()

## Figure 5: Factor Exposure

In [ ]:
from src.viz import plot_factor_exposure

# Run factor regression for best strategy
erc_rets = results['ERC_10'].portfolio_returns.dropna()
aligned = pd.concat([erc_rets, factors[['mkt_rf', 'smb']]], axis=1).dropna()
reg = factor_regression(aligned.iloc[:, 0], aligned[['mkt_rf', 'smb']])

fig = plot_factor_exposure(reg, save_path=str(FIGURES / 'fig5_factors.png'))
plt.show()

## Figure 6: Regime Performance

In [ ]:
from src.viz import plot_regime_performance

regime_dates = {
    '2022 Bear':      ('2022-01-04', '2022-10-31'),
    '2022 Q4 Relief':  ('2022-11-01', '2023-01-31'),
    '2023 Sideways':  ('2023-02-01', '2024-01-31'),
    '2024 Rally':     ('2024-02-01', '2024-10-07'),
    '2024-25 Stable': ('2024-10-08', '2025-12-31'),
}

fig = plot_regime_performance(
    results, regime_dates,
    save_path=str(FIGURES / 'fig6_regimes.png'),
)
plt.show()

## Figure 7: Performance Comparison Table

In [ ]:
from src.viz import compare_to_benchmark_table

fig = compare_to_benchmark_table(
    results,
    benchmark_returns=marp_ret,
    save_path=str(FIGURES / 'fig7_table.png'),
)
plt.show()

## Figure 8: Gold Ablation

In [ ]:
# Gold ablation plot
assets_no_gold = ['510300', '510500', '511010', '159980']
returns_no_gold = returns[assets_no_gold].dropna()

results_no_gold = run_backtest(
    returns_no_gold, marp_returns=marp_ret,
    lookback=756, rebal_days=252,
    strategies={
        'ERC_noGold': lambda r, m: allocate_erc(r, vol_target=0.10),
        'Equal_noGold': lambda r, m: allocate_equal_weight(r),
    }
)

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'ERC_10': '#2196F3', 'ERC_noGold': '#F44336', 'Equal': '#4CAF50', 'Equal_noGold': '#FF9800'}

ax.plot(results['ERC_10'].cumulative.index, results['ERC_10'].cumulative.values,
        color=colors['ERC_10'], linewidth=1.8, label='ERC_10 (with Gold)')
ax.plot(results_no_gold['ERC_noGold'].cumulative.index, results_no_gold['ERC_noGold'].cumulative.values,
        color=colors['ERC_noGold'], linewidth=1.8, linestyle='--', label='ERC_10 (without Gold)')
ax.plot(results['Equal'].cumulative.index, results['Equal'].cumulative.values,
        color=colors['Equal'], linewidth=1.2, label='Equal (with Gold)')
ax.plot(results_no_gold['Equal_noGold'].cumulative.index, results_no_gold['Equal_noGold'].cumulative.values,
        color=colors['Equal_noGold'], linewidth=1.2, linestyle='--', label='Equal (without Gold)')

ax.set_title('Gold Ablation — Impact on Risk-Adjusted Performance', fontsize=13)
ax.set_ylabel('Cumulative Return')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.legend(fontsize=10, ncol=2)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(str(FIGURES / 'fig8_gold_ablation.png'), dpi=200, bbox_inches='tight')
plt.show()

print(f'Gold contribution to ERC_10 total return: {results["ERC_10"].total_return - results_no_gold["ERC_noGold"].total_return:.4f}')
print(f'Gold Sharpe delta: {results["ERC_10"].sharpe - results_no_gold["ERC_noGold"].sharpe:.4f}')

## Figure 9: Strategy Correlation Matrix

In [ ]:
# Cross-strategy correlation
strat_rets = {}
for name, res in results.items():
    strat_rets[name] = res.portfolio_returns.dropna()

corr_df = pd.DataFrame({k: v for k, v in strat_rets.items()}).corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_df, annot=True, fmt='.3f', cmap='RdYlGn', vmin=-1, vmax=1,
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax.set_title('Cross-Strategy Return Correlation', fontsize=13)
fig.tight_layout()
fig.savefig(str(FIGURES / 'fig9_correlation.png'), dpi=200, bbox_inches='tight')
plt.show()

## Figure 10: Annual Return Calendar

In [ ]:
from src.viz import plot_annual_returns_heatmap

erc_rets = results['ERC_10'].portfolio_returns.dropna()
fig = plot_annual_returns_heatmap(
    erc_rets,
    save_path=str(FIGURES / 'fig10_calendar.png'),
)
plt.show()

## All figures saved to `reports/figures/`